# Steer generation with an SAE feature

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/steer_generation.ipynb)

Test a feature intervention against an unsteered baseline and measure the generated sequences.
Outputs: FASTA for each strength, candidate properties, measured feature activations, and comparison plots.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


In [ ]:
SAE = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
FEATURE_ID = 6151 # Example observed in proline-rich sequences; inspect before interpreting
STRENGTHS = [0.0, 0.25, 0.5] # Zero is the baseline; units depend on feature direction magnitude
N = 8
BATCH_SIZE = 2
SEED = 0
TEMPERATURE = 1.0
TOP_P = 0.95
MAX_NEW_TOKENS = 96
OUT_DIR = Path("steering_outputs")


In [ ]:
started = time.perf_counter()


## Generate a baseline and strength sweep

This is an intervention on a model activation, not a biological experiment. Feature IDs belong
to specific SAE weights. With `mode="add_direction"` and `normalize=False`, strength scales the
raw decoder direction; numerical strengths are not directly comparable across features.
The zero-strength condition uses the same generation route and settings. A common seed reduces
one source of variability but does not make samples paired biological replicates.


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
if sae.fim_mode != "unprompted" or sae.region != "idr":
    raise ValueError("Use an unprompted IDR SAE.")
if not 0 <= FEATURE_ID < sae.sae.num_latents:
    raise ValueError("FEATURE_ID outside SAE latent range.")
if not STRENGTHS or 0.0 not in STRENGTHS or len(set(STRENGTHS)) != len(STRENGTHS):
    raise ValueError("Use unique strengths including 0.0 as the baseline.")
generated, rows = [], []
for condition, strength in enumerate(STRENGTHS):
    seqs = sae.steer_generate(feature=FEATURE_ID, strength=strength, n=N,
                              mode="add_direction", normalize=False, seed=SEED,
                              temperature=TEMPERATURE, top_p=TOP_P, batch_size=BATCH_SIZE,
                              max_new_tokens=MAX_NEW_TOKENS)
    recs = [Record(f"condition_{condition}_sample_{i}", s, 0, len(s)) for i, s in enumerate(seqs) if s]
    write_fasta(recs, OUT_DIR / f"condition_{condition}.fasta")
    print(f"Strength {strength}: {len(recs)} nonempty sequences of {N} requested")
    generated.extend(recs)
    rows.extend(dict(record_id=r.accession, strength=strength, sequence=r.full_seq, length=len(r.full_seq),
                     proline_fraction=r.full_seq.count("P") / len(r.full_seq)) for r in recs)
if not generated:
    raise ValueError("All generations were empty; adjust sampling settings.")


## Measure activations without steering hooks

Re-encode the generated sequences normally. Compare peak activation and the fraction of residues
where the target feature fires, alongside length and composition. Higher activation does not
establish the feature's biological meaning, and a response need not be monotonic in strength.
The proline fraction is an illustrative readout for the default feature; choose relevant
sequence properties when changing `FEATURE_ID`.


In [ ]:
from idiom.sae.features import FeatureDataset
check_context(generated, sae.model.cfg.max_seq_len)
sae.build_feature_dataset(generated, OUT_DIR / "features", batch_size=BATCH_SIZE)
dataset = FeatureDataset(OUT_DIR / "features")
_, peaks, fractions = dataset.feature_stats(FEATURE_ID)
results = pd.DataFrame(rows)
results["peak_activation"] = peaks
results["firing_fraction"] = fractions
results.to_csv(OUT_DIR / "candidates.csv", index=False)
results[["record_id", "strength"]].assign(dataset_sequence=np.arange(len(results))).to_csv(
    OUT_DIR / "sequence_index.csv", index=False)
display(results)
fig, axes = plt.subplots(1, 4, figsize=(13, 3), constrained_layout=True)
for ax, metric in zip(axes, ["peak_activation", "firing_fraction", "length", "proline_fraction"]):
    for x, strength in enumerate(STRENGTHS):
        vals = results.loc[results.strength == strength, metric]
        ax.scatter(np.full(len(vals), x), vals, alpha=0.6)
        if len(vals):
            ax.plot([x - 0.2, x + 0.2], [vals.mean()] * 2, color="black")
    ax.set_xticks(range(len(STRENGTHS)), STRENGTHS)
    ax.set(xlabel="Steering strength", ylabel=metric.replace("_", " "))
fig.savefig(OUT_DIR / "steering_comparison.png", dpi=160)
plt.show()
save_run(OUT_DIR, dict(device=str(sae.device), sae=SAE, feature=FEATURE_ID, strengths=STRENGTHS, n=N, seed=SEED,
                       batch_size=BATCH_SIZE, temperature=TEMPERATURE, top_p=TOP_P,
                       max_new_tokens=MAX_NEW_TOKENS, mode="add_direction", normalize=False), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


## Continue the experiment

The black lines show means; individual points show the small sample size. Repeat with additional
seeds and more samples before drawing conclusions. Save the settings for every run in a separate
output directory. Use [compare_sequence_sets.ipynb](compare_sequence_sets.ipynb) to compare a
condition's FASTA with natural reference IDRs, and [inspect_sae_features.ipynb](inspect_sae_features.ipynb)
to inspect the target and other features along individual candidates.
